# UAE Mobile Intelligence - OpenStreetMap Collection

Downloads the Geofabrik GCC States OSM extract (there is no UAE-only
subregion). This notebook only acquires and verifies the raw PBF file.

Filtering it down to UAE buildings, roads, POIs and land-use polygons with
pyosmium (pyrosm is too slow on a file this size) is deferred to a later
processing notebook, once we're ready to build the peer-group density
layers.

Source: https://download.geofabrik.de/asia/gcc-states-latest.osm.pbf
Licence: ODbL (attribution + share-alike)

In [1]:
import hashlib
import urllib.request
from pathlib import Path

In [2]:
OSM_URL = "https://download.geofabrik.de/asia/gcc-states-latest.osm.pbf"

RAW_DIR = Path("../data/raw/osm")
RAW_DIR.mkdir(parents=True, exist_ok=True)

pbf_path = RAW_DIR / "gcc-states-latest.osm.pbf"

In [3]:
# Download (skip if already present). ~250 MB, so this can take a few minutes.

if pbf_path.exists():
    print("Already downloaded:", pbf_path)
else:
    print("Downloading:", OSM_URL)
    urllib.request.urlretrieve(OSM_URL, pbf_path)
    print("Saved to:", pbf_path)

size_mb = pbf_path.stat().st_size / (1024 ** 2)
print(f"File size: {size_mb:.2f} MB")

Downloading: https://download.geofabrik.de/asia/gcc-states-latest.osm.pbf


Saved to: ..\data\raw\osm\gcc-states-latest.osm.pbf
File size: 240.45 MB


In [4]:
# Checksum, for reproducibility record-keeping

sha256 = hashlib.sha256(pbf_path.read_bytes()).hexdigest()
print("SHA256:", sha256)

SHA256: 5897873ba2f12adf85e046bee9093919eaf111f6914d7c4f33cfd6a6059d23c4


In [5]:
# --------------------------------------------
# Basic PBF sanity check: verify the OSM PBF blob header parses
# (a fuller pyosmium-based feature audit is deferred, see markdown above)
# --------------------------------------------

with open(pbf_path, "rb") as f:
    header = f.read(4)

blob_header_len = int.from_bytes(header, byteorder="big")
print("First blob header length:", blob_header_len, "bytes")
assert 0 < blob_header_len < 64 * 1024, "Unexpected PBF blob header size"
print("PBF file structure looks valid.")

First blob header length: 14 bytes
PBF file structure looks valid.


## Notes

- Geofabrik has no UAE-only subregion, so the GCC States extract (covers
  UAE, Saudi Arabia, Oman, Qatar, Bahrain, Kuwait) is the smallest available
  unit. It must be clipped to the UAE boundary polygon downstream, same as
  the Ookla tiles.
- File size (~250 MB) and recency match what the brief describes.
- Next step (not in this notebook): use pyosmium to extract UAE building
  footprints, roads, POIs and land-use polygons, clipped to
  `data/raw/boundary/uae_boundary.geojson`. This feeds the peer-group
  density composite (population density, building-footprint density, POI
  density, road density) described in the brief — do not classify zones by
  OSM land-use tag alone, since commercial/retail tagging is too thin in
  the UAE.
- Licence is ODbL: any published output derived from this data requires
  attribution and share-alike for the derived data.